# 05 — Hyperparameter-Tuning

**Ziel dieses Notebooks:** Die drei Baseline-Modelle aus Notebook 04
(LogReg_balanced, MultinomialNB, LinearSVC) systematisch tunen —
datengetrieben, nicht durch Annahmen. Hauptkandidat ist
LogReg_balanced Config C (F1=0.7775, niedrigste Varianz, hoechster
ROC-AUC). LinearSVC hat das groesste Tuning-Potenzial (Gap=0.181).

**Methodik:**
- Grid Search (systematische Parametersuche) mit Stratified K-Fold CV
- Nested CV: aeussere Schleife fuer unbiasierte Performance-Schaetzung,
  innere Schleife fuer Hyperparameter-Optimierung
- Optimierungsziel: F1 Macro (konsistent mit Phase 4)
- Vergleich: Baseline (ungetunt) vs. getunt — Delta zeigt echten Gewinn

**Suchraum-Prinzip:** Systematisch, aber fokussiert. Jeder Parameter
wird mit klarer Begruendung aufgenommen — kein blindes Durchprobieren.

**Erwarteter Gewinn:** +0.01-0.03 F1 durch Tuning (Bias-Variance
Tradeoff: LinearSVC hat groesstes Potenzial durch Reguliarisierung).

**Output:** getuntes Champion-Modell gespeichert als .joblib,
Tuning-Ergebnisse in reports/tables/, Visualisierungen in reports/figures/

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Daten + Vectorizer laden, Store44-Style aktivieren
# =============================================================================

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_val_score, cross_validate)
from sklearn.metrics import (
    f1_score, roc_auc_score, make_scorer,
    matthews_corrcoef, precision_score, recall_score)

# Datensatz laden
train = pd.read_csv(
    "../data/processed/train_preprocessed.csv",
    keep_default_na=False,
)
train[["keyword", "location"]] = train[["keyword", "location"]].replace("", np.nan)
y = train["target"].values

# Vectorizer laden und Matrix rekonstruieren (Config C = Hauptkandidat)
vectorizer_C = joblib.load("../models/vectorizer_C.joblib")
X = vectorizer_C.transform(train["text_stemmed"])

print("Shape train:", train.shape)
print("Shape X (Config C):", X.shape)
print("Klassenverteilung:", np.bincount(y))

# CV-Setup (konsistent mit Phase 4)
SKF_OUTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SKF_INNER = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Scorer fuer Grid Search
f1_scorer = make_scorer(f1_score, average="macro")

print("\nCV-Setup:")
print(f"  Aeussere CV: k=5 (Performance-Schaetzung)")
print(f"  Innere CV:   k=3 (Hyperparameter-Optimierung)")
print(f"  Scorer:      F1 Macro")

In [ ]:
# =============================================================================
# Zelle 03 – Grid Search Suchraeume definieren
# =============================================================================
# Jeder Parameter wird begruendet aufgenommen — kein blindes Durchprobieren.

# --- LogReg_balanced ---
logreg_param_grid = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    # C = inverse Regularisierungsstaerke:
    # kleines C = starke Regularisierung (weniger Overfitting, mehr Bias)
    # grosses C = schwache Regularisierung (mehr Variance, weniger Bias)
    # Baseline Gap=0.099 → C wahrscheinlich etwas zu gross, kleinere
    # Werte koennen Gap reduzieren und Generalisierung verbessern

    "solver": ["lbfgs", "saga"],
    # lbfgs: Standard, gut fuer kleine-mittlere Datensaetze
    # saga: unterstuetzt L1-Regularisierung (Sparsity),
    #       interessant bei hochdimensionalen TF-IDF-Vektoren

    "penalty": ["l2", "l1"],
    # l2: Standard Ridge-Regularisierung (alle Koeffizienten klein)
    # l1: Lasso-Regularisierung (manche Koeffizienten exakt 0 = Feature Selection)
    # Hinweis: l1 nur mit solver="saga" kompatibel
}

# --- MultinomialNB ---
nb_param_grid = {
    "alpha": [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    # alpha = Laplace-Glaettung (additive Glaettung):
    # verhindert Wahrscheinlichkeit=0 fuer unsehene Woerter
    # kleines alpha = wenig Glaettung (staerkeres Signal aus Daten)
    # grosses alpha = starke Glaettung (aehnlicher zu Gleichverteilung)
    # Standard: alpha=1.0 (Laplace), Baseline-Gap=0.077 → alpha vermutlich gut
    # aber systematisch testen ob kleiner/groesseres alpha hilft
}

# --- LinearSVC ---
svc_param_grid = {
    "C": [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
    # Grosses Tuning-Potenzial: Baseline-Gap=0.181 (stark overfitting)
    # Erwartung: sehr kleines C (starke Regularisierung) bringt grossen Gewinn
    # Aktueller Default C=1.0 ist zu gross fuer diesen Datensatz

    "loss": ["hinge", "squared_hinge"],
    # hinge: klassische SVM-Verlustfunktion (original Margin-Optimierung)
    # squared_hinge: quadratischer Verlust, glattere Optimierung,
    #               oft etwas bessere Performance bei Text-Daten
}

print("=== Suchraeume ===\n")
print("LogReg_balanced:")
for param, values in logreg_param_grid.items():
    print(f"  {param}: {values}")
total_logreg = 1
for v in logreg_param_grid.values():
    total_logreg *= len(v)
print(f"  → {total_logreg} Kombinationen × 3 (innere CV) = "
      f"{total_logreg * 3} Fits")

print("\nMultinomialNB:")
for param, values in nb_param_grid.items():
    print(f"  {param}: {values}")
print(f"  → {len(nb_param_grid['alpha'])} Kombinationen × 3 = "
      f"{len(nb_param_grid['alpha']) * 3} Fits")

print("\nLinearSVC:")
for param, values in svc_param_grid.items():
    print(f"  {param}: {values}")
total_svc = 1
for v in svc_param_grid.values():
    total_svc *= len(v)
print(f"  → {total_svc} Kombinationen × 3 (innere CV) = "
      f"{total_svc * 3} Fits")

## Methodik: Grid Search + Nested Cross-Validation

**Grid Search:**
Systematisches Durchprobieren aller Parameter-Kombinationen im
definierten Suchraum. Fuer jede Kombination wird die innere CV
(k=3) ausgefuehrt und der mittlere F1-Score berechnet. Die
Kombination mit dem hoechsten F1 wird als "beste Parameter" gewaehlt.

**Warum Nested CV (zwei verschachtelte Schleifen)?**
Ein klassisches Problem: wenn man Hyperparameter auf denselben Daten
sucht UND evaluiert, ueberschaetzt man die Performance ("Optimismus-
Bias"). Das Modell "weiss" bereits, welche Parameter auf diesen Daten
gut funktionieren.

Loesung — zwei Ebenen:
- **Innere Schleife (k=3):** findet beste Hyperparameter
- **Aeussere Schleife (k=5):** evaluiert Performance mit den in diesem
  Fold gefundenen Parametern auf UNGESEHENEN Daten

Ergebnis: 5 unabhaengige Performance-Schaetzungen mit je optimal
getunten Parametern → unbiasierter F1-Mittelwert.

**Was die Parameter bedeuten:**

LogReg — C (Regularisierungsstaerke):
  Kleines C (0.001):  viele Koeffizienten nahe 0, Modell ignoriert
                      seltene Woerter → weniger Overfitting, aber
                      moeglicherweise wichtige Features verloren
  Grosses C (100.0):  alle Koeffizienten frei → maximale Anpassung
                      an Trainingsdaten → Overfitting-Risiko

LogReg — penalty (L1 vs. L2):
  L2 (Ridge): alle Koeffizienten werden klein, keiner exakt 0
  L1 (Lasso): manche Koeffizienten werden exakt 0 → automatische
              Feature Selection (irrelevante Woerter = Gewicht 0)
              Besonders interessant bei 4.869 TF-IDF-Dimensionen

NB — alpha (Laplace-Glaettung):
  Verhindert P(Wort|Klasse) = 0 fuer unsehene Woerter.
  Kleines alpha: Daten dominieren (scharfe Entscheidungen)
  Grosses alpha: Glaettung dominiert (konservativere Entscheidungen)

LinearSVC — C (Margin-Weite):
  Kleines C: breiter Margin, mehr Fehlklassifikationen erlaubt
             → weniger Overfitting (Hauptziel hier: Gap 0.181 → < 0.05)
  Grosses C: enger Margin, kaum Fehler im Training erlaubt
             → Overfitting (aktuelles Problem)

**Gesamtaufwand:**
  LogReg:    72 Fits × 5 (aeussere CV) = 360 Fits
  NB:        24 Fits × 5 (aeussere CV) = 120 Fits
  LinearSVC: 36 Fits × 5 (aeussere CV) = 180 Fits
  Gesamt:    660 Fits (CPU, geschaetzte Laufzeit: 3-8 Minuten)

In [ ]:
# =============================================================================
# Zelle 05 – LogReg_balanced: Grid Search + Nested CV
# =============================================================================

from sklearn.model_selection import GridSearchCV, cross_val_score

print("LogReg_balanced: Grid Search + Nested CV")
print("Laufzeit: ~2-4 Minuten...\n")

# Innere Schleife: Grid Search (findet beste Parameter)
logreg_base = LogisticRegression(
    class_weight="balanced", random_state=42, max_iter=2000)

logreg_grid = GridSearchCV(
    estimator=logreg_base,
    param_grid=logreg_param_grid,
    cv=SKF_INNER,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=0,
)

# Aeussere Schleife: Nested CV (unbiasierte Performance-Schaetzung)
nested_scores_logreg = cross_val_score(
    logreg_grid, X, y,
    cv=SKF_OUTER,
    scoring=f1_scorer,
    n_jobs=-1,
)

print(f"Nested CV F1 (5 Folds): {nested_scores_logreg}")
print(f"Nested CV F1 Mittelwert: {nested_scores_logreg.mean():.4f} "
      f"± {nested_scores_logreg.std():.4f}")

# Beste Parameter auf gesamtem Datensatz finden
logreg_grid.fit(X, y)
print(f"\nBeste Parameter (auf gesamtem Datensatz):")
print(f"  {logreg_grid.best_params_}")
print(f"  Bester innerer CV F1: {logreg_grid.best_score_:.4f}")

# Vergleich: Baseline vs. getunt
baseline_f1 = 0.7775
tuned_f1 = nested_scores_logreg.mean()
print(f"\n=== Vergleich Baseline vs. Getunt ===")
print(f"  Baseline F1 (Phase 4):  {baseline_f1:.4f}")
print(f"  Getunt F1 (Nested CV):  {tuned_f1:.4f}")
print(f"  Delta:                  {tuned_f1 - baseline_f1:+.4f}")

# Ergebnistabelle speichern
logreg_tuning_results = pd.DataFrame(logreg_grid.cv_results_)
logreg_tuning_results.to_csv(
    "../reports/tables/05_logreg_tuning_results.csv", index=False)

# Besten LogReg speichern (temporaer, wird am Ende durch Champion ersetzt)
best_logreg_tuned = logreg_grid.best_estimator_
joblib.dump(best_logreg_tuned, "../models/tuned_logreg.joblib")
print("\nBestes Modell gespeichert: models/tuned_logreg.joblib")

## Erkenntnis: LogReg_balanced Tuning

**Nested CV Ergebnis:**
  F1 pro Fold: [0.7709, 0.7834, 0.7889, 0.7757, 0.7724]
  F1 Mittelwert: 0.7782 ± 0.0068
  Beste Parameter: C=1.0, penalty=L2, solver=lbfgs

**Vergleich Baseline vs. Getunt:**
  Baseline (Phase 4): 0.7775
  Getunt (Nested CV): 0.7782
  Delta: +0.0007 — marginale Verbesserung

**Kernbefunde:**

1. **Default-Parameter waren bereits optimal:**
   Grid Search bestaetigt C=1.0, L2, lbfgs als beste Kombination —
   genau die sklearn-Standardwerte. Das bedeutet: LogReg war bereits
   in Phase 4 praktisch optimal konfiguriert. Tuning bringt hier
   keinen echten Gewinn.

2. **L1-Regularisierung hilft nicht:**
   Trotz 4.869 TF-IDF-Dimensionen profitiert LogReg nicht von
   L1-Feature-Selection (Lasso). Erklaerung: TF-IDF gewichtet bereits
   selten vorkommende, wenig informative Woerter niedrig (IDF-Term) —
   L2 reicht, da L1 keine zusaetzliche Selektion benoetigt.

3. **Stabilitaet bestaetigt:** Std 0.0068 (konsistent mit Phase 4:
   0.0069) — das Modell ist robust gegenueber Datensplit-Variationen.
   Nested CV zeigt keinen systematischen Optimismus-Bias in Phase 4
   (Delta nur +0.0007 statt z.B. +0.03, was auf Bias hindeuten wuerde).

4. **Interpretation des kleinen Deltas:**
   +0.0007 liegt weit unterhalb der Standardabweichung (0.0068) —
   statistisch nicht signifikant. LogReg hat sein Maximum mit diesem
   Datensatz und dieser Vektorisierung erreicht. Weiteres Tuning wuerde
   keinen messbaren Mehrwert bringen.

**Konsequenz:**
LogReg_balanced mit Default-Parametern (C=1.0, L2, lbfgs) bleibt
Hauptkandidat. Das Tuning-Ergebnis ist inhaltlich wertvoll: es zeigt,
dass die Baseline-Ergebnisse aus Phase 4 zuverlaessig waren und
nicht durch unguenstige Hyperparameter verzerrt wurden.

In [ ]:
# =============================================================================
# Zelle 07 – MultinomialNB: Grid Search + Nested CV
# =============================================================================

print("MultinomialNB: Grid Search + Nested CV")
print("Laufzeit: ~1 Minute...\n")

nb_base = MultinomialNB()

nb_grid = GridSearchCV(
    estimator=nb_base,
    param_grid=nb_param_grid,
    cv=SKF_INNER,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=0,
)

# Nested CV
nested_scores_nb = cross_val_score(
    nb_grid, X, y,
    cv=SKF_OUTER,
    scoring=f1_scorer,
    n_jobs=-1,
)

print(f"Nested CV F1 (5 Folds): {nested_scores_nb}")
print(f"Nested CV F1 Mittelwert: {nested_scores_nb.mean():.4f} "
      f"± {nested_scores_nb.std():.4f}")

# Beste Parameter auf gesamtem Datensatz
nb_grid.fit(X, y)
print(f"\nBeste Parameter (auf gesamtem Datensatz):")
print(f"  {nb_grid.best_params_}")
print(f"  Bester innerer CV F1: {nb_grid.best_score_:.4f}")

# Vergleich
baseline_nb_f1 = 0.7799
tuned_nb_f1 = nested_scores_nb.mean()
print(f"\n=== Vergleich Baseline vs. Getunt ===")
print(f"  Baseline F1 (Phase 4):  {baseline_nb_f1:.4f}")
print(f"  Getunt F1 (Nested CV):  {tuned_nb_f1:.4f}")
print(f"  Delta:                  {tuned_nb_f1 - baseline_nb_f1:+.4f}")

# Alle alpha-Werte und ihre CV-Scores visualisieren
nb_cv_results = pd.DataFrame(nb_grid.cv_results_)
nb_cv_results.to_csv(
    "../reports/tables/05_nb_tuning_results.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(nb_cv_results["param_alpha"],
        nb_cv_results["mean_test_score"],
        marker="o", color=COLOR_GOLD, linewidth=2)
ax.fill_between(
    nb_cv_results["param_alpha"],
    nb_cv_results["mean_test_score"] - nb_cv_results["std_test_score"],
    nb_cv_results["mean_test_score"] + nb_cv_results["std_test_score"],
    alpha=0.2, color=COLOR_GOLD)
ax.set_xscale("log")
ax.set_xlabel("alpha (log-Skala)")
ax.set_ylabel("F1 Macro (innere CV, k=3)")
ax.set_title("MultinomialNB: F1 vs. alpha")
ax.axvline(nb_grid.best_params_["alpha"], color=COLOR_BLUE,
           linestyle="--", label=f"Bestes alpha={nb_grid.best_params_['alpha']}")
ax.legend()
fig.tight_layout()

save_figure(fig, "../reports/figures/05_nb_alpha_tuning.png")
plt.show()

best_nb_tuned = nb_grid.best_estimator_
joblib.dump(best_nb_tuned, "../models/tuned_nb.joblib")
print("\nBestes Modell gespeichert: models/tuned_nb.joblib")

## Erkenntnis: MultinomialNB Tuning

**Nested CV Ergebnis:**
  F1 pro Fold: [0.7614, 0.7884, 0.7862, 0.7734, 0.7837]
  F1 Mittelwert: 0.7786 ± 0.0100
  Bestes alpha: 0.5 (Standard: 1.0)

**Vergleich Baseline vs. Getunt:**
  Baseline F1 (Phase 4): 0.7799
  Getunt F1 (Nested CV): 0.7786
  Delta: -0.0013

**Kernbefunde:**

1. **Tuning bringt keinen Gewinn — leichter Rueckgang:**
   Der negative Delta (-0.0013) ist statistisch nicht signifikant
   (weit unterhalb der Std 0.0100). Der Nested-CV-Wert liegt minimal
   unter der Baseline, weil Nested CV strenger schaetzt (kein
   Optimismus-Bias) — die "wahre" Performance ist etwas niedriger
   als die einfache CV aus Phase 4.

2. **alpha-Kurve zeigt klares Optimum bei 0.5:**
   - Zu wenig Glaettung (alpha < 0.1): seltene Woerter uebergewichtet
     → Overfitting auf Trainingsvokabular
   - Zu viel Glaettung (alpha > 2): Disaster-Signalwoerter verlieren
     ihre Unterscheidungskraft → Modell wird zu konservativ
   - alpha=0.5: optimale Balance, aber Verbesserung ggue. alpha=1.0
     ist marginal (+0.004 innere CV F1)

3. **Stabilitaet bestaetigt (Std 0.0100):** Konsistent mit Phase 4.
   Schmales Konfidenzband im Plot bestaetigt: NB ist robust
   gegenueber Datensplit-Variationen.

4. **NB hat wenig Tuning-Potenzial:**
   Nur ein relevanter Hyperparameter (alpha), und dessen Effekt ist
   begrenzt. NB ist ein einfaches, direktes Modell — seine Grenzen
   liegen in der Modellklasse (Naive-Bayes-Unabhaengigkeitsannahme),
   nicht in den Hyperparametern.

**Konsequenz:** MultinomialNB mit alpha=0.5 als Vergleichskandidat
behalten. Kein substantieller Gewinn durch Tuning erwartet oder
erzielt — bestaetigt, dass NB sein Maximum erreicht hat.

In [ ]:
# =============================================================================
# Zelle 09 – LinearSVC: Grid Search + Nested CV
# =============================================================================
# Groesstes Tuning-Potenzial: Baseline-Gap=0.181, Train-F1=0.949
# Erwartung: starke Regularisierung (kleines C) reduziert Gap erheblich

print("LinearSVC: Grid Search + Nested CV")
print("Laufzeit: ~2-3 Minuten...\n")

svc_base = LinearSVC(max_iter=5000, random_state=42)

svc_grid = GridSearchCV(
    estimator=svc_base,
    param_grid=svc_param_grid,
    cv=SKF_INNER,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=0,
)

# Nested CV
nested_scores_svc = cross_val_score(
    svc_grid, X, y,
    cv=SKF_OUTER,
    scoring=f1_scorer,
    n_jobs=-1,
)

print(f"Nested CV F1 (5 Folds): {nested_scores_svc}")
print(f"Nested CV F1 Mittelwert: {nested_scores_svc.mean():.4f} "
      f"± {nested_scores_svc.std():.4f}")

# Beste Parameter auf gesamtem Datensatz
svc_grid.fit(X, y)
print(f"\nBeste Parameter (auf gesamtem Datensatz):")
print(f"  {svc_grid.best_params_}")
print(f"  Bester innerer CV F1: {svc_grid.best_score_:.4f}")

# Vergleich
baseline_svc_f1 = 0.7681
tuned_svc_f1 = nested_scores_svc.mean()
print(f"\n=== Vergleich Baseline vs. Getunt ===")
print(f"  Baseline F1 (Phase 4):  {baseline_svc_f1:.4f}")
print(f"  Getunt F1 (Nested CV):  {tuned_svc_f1:.4f}")
print(f"  Delta:                  {tuned_svc_f1 - baseline_svc_f1:+.4f}")

# C-Werte vs. F1 visualisieren (fuer beide loss-Varianten)
svc_cv_results = pd.DataFrame(svc_grid.cv_results_)
svc_cv_results.to_csv(
    "../reports/tables/05_svc_tuning_results.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for loss_val, color, label in [
    ("hinge", COLOR_GOLD, "hinge"),
    ("squared_hinge", COLOR_BLUE, "squared_hinge")
]:
    subset = svc_cv_results[
        svc_cv_results["param_loss"] == loss_val
    ].sort_values("param_C")
    ax.plot(subset["param_C"], subset["mean_test_score"],
            marker="o", color=color, linewidth=2, label=label)
    ax.fill_between(
        subset["param_C"],
        subset["mean_test_score"] - subset["std_test_score"],
        subset["mean_test_score"] + subset["std_test_score"],
        alpha=0.2, color=color)

ax.set_xscale("log")
ax.set_xlabel("C (log-Skala)")
ax.set_ylabel("F1 Macro (innere CV, k=3)")
ax.set_title("LinearSVC: F1 vs. C (je loss-Funktion)")
ax.axvline(svc_grid.best_params_["C"], color=COLOR_TEXT_MUTED,
           linestyle="--",
           label=f"Bestes C={svc_grid.best_params_['C']}")
ax.legend()
fig.tight_layout()

save_figure(fig, "../reports/figures/05_svc_c_tuning.png")
plt.show()

# Overfitting-Gap des besten getunten SVC messen
best_svc_tuned = svc_grid.best_estimator_

# Gap auf vollem Datensatz schaetzen
train_f1_tuned = f1_score(y, best_svc_tuned.predict(X), average="macro")
cv_f1_tuned = nested_scores_svc.mean()
print(f"\nOverfitting-Gap (getunt):")
print(f"  Train-F1: {train_f1_tuned:.4f}")
print(f"  CV-F1:    {cv_f1_tuned:.4f}")
print(f"  Gap:      {train_f1_tuned - cv_f1_tuned:.4f} "
      f"(Baseline-Gap war: 0.181)")

joblib.dump(best_svc_tuned, "../models/tuned_svc.joblib")
print("\nBestes Modell gespeichert: models/tuned_svc.joblib")

## Erkenntnis: LinearSVC Tuning

**Nested CV Ergebnis:**
  F1 pro Fold: [0.7733, 0.7854, 0.7849, 0.7835, 0.7765]
  F1 Mittelwert: 0.7807 ± 0.0049
  Beste Parameter: C=1.0, loss=hinge

**Vergleich Baseline vs. Getunt:**
  Baseline F1 (Phase 4):  0.7681
  Getunt F1 (Nested CV):  0.7807
  Delta: +0.0126 — groesster Tuning-Gewinn aller drei Modelle

**Overfitting-Gap:**
  Baseline: Train-F1=0.949, Gap=0.181
  Getunt:   Train-F1=0.887, Gap=0.106 (-42% Reduktion)

**Kernbefunde:**

1. **Erwartung bestaetigt: LinearSVC hatte groesstes Tuning-Potenzial**
   +0.0126 F1-Gewinn durch Tuning (vs. +0.0007 LogReg, -0.0013 NB).
   Der grosse Baseline-Gap (0.181) war ein zuverlaessiger Indikator
   fuer verbleibendes Verbesserungspotenzial — Bias-Variance-Tradeoff
   Theorie bestaetigt sich empirisch.

2. **Ueberraschendes Ergebnis: Bestes C=1.0 (Standard-Wert)**
   Trotz massivem Baseline-Overfitting wurde C=1.0 als optimal
   gefunden — nicht das erwartete sehr kleine C (0.001-0.01).
   Erklaerung: Das Baseline-Overfitting in Phase 4 kam nicht von
   einem zu grossen C, sondern von der Kombination C=1.0 +
   DEFAULT max_iter=1000 (Konvergenzproblem). Mit max_iter=5000
   hier konvergiert das Modell sauber — und C=1.0 ist tatsaechlich
   optimal.

3. **Plot zeigt kritisches C-Verhalten:**
   Sehr kleines C (< 0.01): F1 kollabiert auf ~0.38 — zu starke
   Regularisierung macht das Modell nutzlos (alle Koeffizienten
   fast 0, Modell sagt immer dieselbe Klasse)
   C=0.1-1.0: F1 steigt steil an — der produktive Bereich
   C > 1.0: leichter Rueckgang — zu wenig Regularisierung

4. **squared_hinge leicht besser bei kleinen C-Werten:**
   Bei C=0.01 und C=0.1 hat squared_hinge etwas hoehere F1
   (0.55 vs. 0.38 bei C=0.01) — bei C=1.0 sind beide fast
   gleich. Hinge gewinnt minimal bei optimalem C.

5. **Gap 0.106 noch nicht ideal, aber deutlich verbessert:**
   Zielbereich waere Gap < 0.05 (sehr gute Generalisierung).
   0.106 bedeutet noch immer messbares Overfitting — aber das
   ist die strukturelle Grenze dieses Modells auf diesem
   Datensatz (kurze Tweets, viele Metaphern).

**LinearSVC getunt: neues bestes Modell (F1=0.7807)**
Vergleich der drei getunten Modelle:
  LinearSVC (getunt):    0.7807 ± 0.0049
  LogReg_balanced:       0.7782 ± 0.0068
  MultinomialNB (getunt): 0.7786 ± 0.0100

In [ ]:
# =============================================================================
# Zelle 11 (korrigiert) – Gesamtvergleich mit ALLEN Metriken
# =============================================================================

from sklearn.metrics import matthews_corrcoef, precision_score, recall_score

# evaluate_model Funktion aus Notebook 04 (identisch, damit Metriken vergleichbar)
def evaluate_model_full(model, X, y, model_name, skf=SKF_OUTER, n_bootstrap=500):
    fold_metrics = {
        "f1_macro": [], "f1_0": [], "f1_1": [],
        "precision_1": [], "recall_1": [],
        "roc_auc": [], "mcc": [],
        "f1_macro_train": []
    }
    oof_preds = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y))

    for train_idx, val_idx in skf.split(X, y):
        if hasattr(X, "toarray"):
            X_tr, X_val = X[train_idx], X[val_idx]
        else:
            X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        y_score = (model.predict_proba(X_val)[:, 1]
                   if hasattr(model, "predict_proba")
                   else model.decision_function(X_val))

        oof_preds[val_idx] = y_pred
        oof_scores[val_idx] = y_score

        fold_metrics["f1_macro"].append(f1_score(y_val, y_pred, average="macro"))
        fold_metrics["f1_0"].append(f1_score(y_val, y_pred, pos_label=0))
        fold_metrics["f1_1"].append(f1_score(y_val, y_pred, pos_label=1))
        fold_metrics["precision_1"].append(precision_score(y_val, y_pred, zero_division=0))
        fold_metrics["recall_1"].append(recall_score(y_val, y_pred, zero_division=0))
        fold_metrics["roc_auc"].append(roc_auc_score(y_val, y_score))
        fold_metrics["mcc"].append(matthews_corrcoef(y_val, y_pred))
        fold_metrics["f1_macro_train"].append(
            f1_score(y_tr, model.predict(X_tr), average="macro"))

    results = {"model": model_name}
    for metric, values in fold_metrics.items():
        results[f"{metric}_mean"] = np.mean(values)
        results[f"{metric}_std"] = np.std(values)
    results["overfitting_gap"] = (
        results["f1_macro_train_mean"] - results["f1_macro_mean"])

    # Bootstrap KI
    rng = np.random.RandomState(42)
    boot_f1, boot_roc = [], []
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        boot_f1.append(f1_score(y[idx], oof_preds[idx], average="macro"))
        boot_roc.append(roc_auc_score(y[idx], oof_scores[idx]))

    results["f1_macro_ci_low"] = np.percentile(boot_f1, 2.5)
    results["f1_macro_ci_high"] = np.percentile(boot_f1, 97.5)
    results["roc_auc_ci_low"] = np.percentile(boot_roc, 2.5)
    results["roc_auc_ci_high"] = np.percentile(boot_roc, 97.5)

    return results

# Alle drei getunten Modelle vollstaendig evaluieren
print("Vollstaendige Evaluation aller getunten Modelle...")
print("(Laufzeit: ~2-3 Minuten)\n")

tuned_models = [
    ("LogReg_balanced_tuned", joblib.load("../models/tuned_logreg.joblib")),
    ("MultinomialNB_tuned", joblib.load("../models/tuned_nb.joblib")),
    ("LinearSVC_tuned", joblib.load("../models/tuned_svc.joblib")),
]

tuned_results = []
for name, model in tuned_models:
    print(f"Evaluiere {name}...", end=" ")
    result = evaluate_model_full(model, X, y, name)
    tuned_results.append(result)
    print(f"F1={result['f1_macro_mean']:.4f} | "
          f"MCC={result['mcc_mean']:.4f} | "
          f"Gap={result['overfitting_gap']:.4f}")

tuned_df = pd.DataFrame(tuned_results)

# Baseline beste Kombination pro Modell aus Phase 4 laden
baseline_all = pd.read_csv("../reports/tables/04_all_results.csv")
baseline_best = {}
for model_names, label in [
    (["LogReg_balanced"], "LogReg_balanced_baseline"),
    (["MultinomialNB"], "MultinomialNB_baseline"),
    (["LinearSVC"], "LinearSVC_baseline"),
]:
    subset = baseline_all[baseline_all["model"].isin(model_names)]
    best = subset.loc[subset["f1_macro_mean"].idxmax()].copy()
    best["model"] = label
    baseline_best[label] = best

baseline_df = pd.DataFrame(baseline_best.values())

# Vollstaendiger Vergleich: alle Metriken
metrics_to_compare = [
    "f1_macro_mean", "f1_0_mean", "f1_1_mean",
    "precision_1_mean", "recall_1_mean",
    "roc_auc_mean", "mcc_mean", "overfitting_gap"
]

print("\n=== BASELINE (beste Kombination Phase 4) ===")
print(baseline_df[["model"] + metrics_to_compare].to_string(index=False))

print("\n=== GETUNT (Phase 5) ===")
print(tuned_df[["model"] + metrics_to_compare].to_string(index=False))

# Delta-Tabelle
print("\n=== DELTA (Getunt - Baseline) ===")
for (bl_name, bl_row), (t_row) in zip(
    baseline_best.items(), tuned_results
):
    print(f"\n{bl_name.replace('_baseline','')}:")
    for m in metrics_to_compare:
        delta = t_row[m] - bl_row[m]
        sign = "+" if delta >= 0 else ""
        print(f"  {m:25s}: {bl_row[m]:.4f} → {t_row[m]:.4f} "
              f"({sign}{delta:.4f})")

# Speichern
tuned_df.to_csv("../reports/tables/05_tuned_full_metrics.csv", index=False)
print("\nGespeichert: reports/tables/05_tuned_full_metrics.csv")

In [ ]:
# =============================================================================
# Zelle 11b – Visualisierung: vollstaendiger Metriken-Vergleich
# =============================================================================

metrics_plot = {
    "F1 Macro": ("f1_macro_mean", "f1_macro_std"),
    "F1 Klasse 1": ("f1_1_mean", "f1_1_std"),
    "Precision_1": ("precision_1_mean", "precision_1_std"),
    "Recall_1": ("recall_1_mean", "recall_1_std"),
    "ROC-AUC": ("roc_auc_mean", "roc_auc_std"),
    "MCC": ("mcc_mean", "mcc_std"),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

model_labels = ["LogReg\nbalanced", "Naive\nBayes", "Linear\nSVC"]
x = np.arange(3)
width = 0.35

for ax, (metric_name, (mean_col, std_col)) in zip(axes, metrics_plot.items()):
    bl_means = baseline_df[mean_col].values
    bl_stds = baseline_df[std_col].values if std_col in baseline_df else np.zeros(3)
    t_means = tuned_df[mean_col].values
    t_stds = tuned_df[std_col].values if std_col in tuned_df else np.zeros(3)

    ax.bar(x - width/2, bl_means, width, label="Baseline",
           color=COLOR_BLUE, alpha=0.85,
           yerr=bl_stds, capsize=3,
           error_kw={"ecolor": COLOR_TEXT_MUTED})
    ax.bar(x + width/2, t_means, width, label="Getunt",
           color=COLOR_GOLD, alpha=0.85,
           yerr=t_stds, capsize=3,
           error_kw={"ecolor": COLOR_TEXT_MUTED})

    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, fontsize=9)
    ax.set_title(metric_name)
    ax.set_ylim(max(0, min(bl_means.min(), t_means.min()) - 0.05),
                max(bl_means.max(), t_means.max()) + 0.05)
    if ax == axes[0]:
        ax.legend(fontsize=8)

fig.suptitle("Phase 5: Alle Metriken — Baseline vs. Getunt\n"
             "Champion: LinearSVC getunt (groesste Verbesserung ueber alle Metriken)")
fig.tight_layout()

save_figure(fig, "../reports/figures/05_all_metrics_comparison.png")
plt.show()

# Overfitting-Gap separat
fig2, ax2 = plt.subplots(figsize=(8, 5))
gap_bl = baseline_df["overfitting_gap"].values
gap_t = tuned_df["overfitting_gap"].values

ax2.bar(x - width/2, gap_bl, width, label="Baseline",
        color=COLOR_BLUE, alpha=0.85)
ax2.bar(x + width/2, gap_t, width, label="Getunt",
        color=COLOR_GOLD, alpha=0.85)
ax2.axhline(0.05, color=COLOR_GREEN, linestyle="--",
            linewidth=1.5, label="Ziel Gap < 0.05")
ax2.set_xticks(x)
ax2.set_xticklabels(model_labels)
ax2.set_ylabel("Overfitting-Gap (Train-F1 - CV-F1)")
ax2.set_title("Overfitting-Gap: Baseline vs. Getunt")
ax2.legend()
fig2.tight_layout()

save_figure(fig2, "../reports/figures/05_overfitting_gap.png")
plt.show()

# Speichern
comparison_df = pd.concat([baseline_df, tuned_df], ignore_index=True)
comparison_df.to_csv("../reports/tables/05_baseline_vs_tuned.csv", index=False)
print("Gespeichert: reports/tables/05_baseline_vs_tuned.csv")

## Erkenntnis: Gesamtvergleich Baseline vs. Getunt

### Ergebnisse alle getunten Modelle

| Modell | F1 Macro | Std | ROC-AUC | MCC | P1 | R1 | Gap |
|---|---|---|---|---|---|---|---|
| LogReg_balanced | 0.7775 | **0.0069** | **0.8487** | 0.5551 | 0.741 | **0.728** | 0.099 |
| MultinomialNB | 0.7802 | 0.0100 | 0.8429 | 0.5702 | **0.803** | 0.658 | **0.090** |
| LinearSVC | **0.7807** | **0.0049** | 0.8424 | 0.5682 | 0.791 | 0.672 | 0.115 |

### Kritische Analyse: Unterschied vs. Varianz

**Alle F1-Unterschiede sind statistisch nicht signifikant:**
- LogReg vs. NB:  Differenz 0.0027 = 0.32× Pooled-Std → Zufall
- LogReg vs. SVC: Differenz 0.0032 = 0.54× Pooled-Std → Zufall
- NB vs. SVC:     Differenz 0.0005 = 0.07× Pooled-Std → Zufall
- 95% KI ueberlappen vollstaendig bei allen drei Modellen

**Ein Ingenieur der eines dieser Modelle auf Basis von F1-Macro
allein als "klar besser" bezeichnet irrt sich — die Unterschiede
liegen im statistischen Rauschen.**

### Entscheidungsgrundlage: Nicht-F1-Kriterien

Die relevanten Unterschiede liegen nicht im F1-Macro:

1. **ROC-AUC (schwellenwert-unabhaengig, robuster):**
   LogReg: 0.8487 > NB: 0.8429 > SVC: 0.8424
   Einzige Metrik mit konsistentem, wenn auch kleinem Vorsprung.

2. **Fehler-Profil (einziger harter, nicht-rauschbedingter Unterschied):**
   LogReg:  P1=0.741, R1=0.728 → ausgewogen
   NB:      P1=0.803, R1=0.658 → 36% Disasters verpasst (FN)
   SVC:     P1=0.791, R1=0.672 → 33% Disasters verpasst (FN)
   Im Disaster-Response-Kontext ist FN (verpasste Katastrophe)
   teurer als FP (Fehlalarm) → ausgewogenes Profil bevorzugt.

3. **Stabilitaet:** LinearSVC (CV=0.63%) > LogReg (0.89%) > NB (1.28%)

4. **Erklaerbarkeit:** LogReg einzigartig — Koeffizienten pro Wort
   direkt interpretierbar ohne SHAP. Relevant fuer Vertrauen und
   Fehleranalyse in Phase 6.

5. **Tuning-Robustheit:** LogReg Default-Parameter waren bereits
   optimal — kein Selektions-Bias durch Grid Search.

### Champion: LogReg_balanced Config C

Begruendung: hoechster ROC-AUC, ausgewogenstes Fehler-Profil,
beste Erklaerbarkeit, robust gegenueber Tuning. F1-Macro-Unterschiede
zu den anderen Modellen liegen im statistischen Rauschen und sind
nicht entscheidungsrelevant.

### Tuning-Bilanz

| Modell | Delta F1 | Erkenntnis |
|---|---|---|
| LogReg_balanced | +0.0007 | Default-Parameter optimal — Tuning unnoetig |
| MultinomialNB | +0.0003 | Marginale Verbesserung F1_1, MCC sinkt |
| LinearSVC | **+0.0126** | Groesster Gewinn: alle Metriken verbessert, Gap -0.066 |

LinearSVC profitiert am staerksten von Tuning — bestaetigt die
Bias-Variance-Tradeoff-Theorie: groesster Baseline-Gap (0.181)
signalisiert groesstes Verbesserungspotenzial.

In [ ]:
# =============================================================================
# Zelle 13 – Threshold-Optimierung: Champion LogReg_balanced Config C
# =============================================================================
# Standard-Schwellenwert 0.5: wenn P(Disaster) > 0.5 → Disaster
# Frage: gibt es einen besseren Schwellenwert fuer unseren Anwendungsfall?
# Methode: PR-Kurve + F1-Kurve + MCC-Kurve ueber verschiedene Schwellenwerte

from sklearn.metrics import (
    precision_recall_curve, f1_score,
    matthews_corrcoef, confusion_matrix
)

# Out-of-Fold Wahrscheinlichkeiten fuer LogReg Champion
champion = LogisticRegression(
    max_iter=1000, random_state=42, class_weight="balanced")

oof_probs = np.zeros(len(y))
oof_preds_default = np.zeros(len(y), dtype=int)

for train_idx, val_idx in SKF_OUTER.split(X, y):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    champion.fit(X_tr, y_tr)
    oof_probs[val_idx] = champion.predict_proba(X_val)[:, 1]
    oof_preds_default[val_idx] = champion.predict(X_val)

# F1, MCC, Precision, Recall ueber Schwellenwerte
thresholds = np.linspace(0.1, 0.9, 81)
f1_scores, mcc_scores = [], []
precision_scores, recall_scores = [], []

for t in thresholds:
    preds = (oof_probs >= t).astype(int)
    if len(np.unique(preds)) < 2:
        f1_scores.append(np.nan)
        mcc_scores.append(np.nan)
        precision_scores.append(np.nan)
        recall_scores.append(np.nan)
        continue
    f1_scores.append(f1_score(y, preds, average="macro"))
    mcc_scores.append(matthews_corrcoef(y, preds))
    precision_scores.append(precision_score(y, preds, zero_division=0))
    recall_scores.append(recall_score(y, preds, zero_division=0))

f1_arr = np.array(f1_scores)
mcc_arr = np.array(mcc_scores)

# Optimale Schwellenwerte
t_best_f1 = thresholds[np.nanargmax(f1_arr)]
t_best_mcc = thresholds[np.nanargmax(mcc_arr)]

print(f"Standard-Schwellenwert (0.50): "
      f"F1={f1_score(y, oof_preds_default, average='macro'):.4f} | "
      f"MCC={matthews_corrcoef(y, oof_preds_default):.4f}")
print(f"Optimaler F1-Schwellenwert:    {t_best_f1:.2f} → "
      f"F1={np.nanmax(f1_arr):.4f}")
print(f"Optimaler MCC-Schwellenwert:   {t_best_mcc:.2f} → "
      f"MCC={np.nanmax(mcc_arr):.4f}")

# Vergleichstabelle: 3 Schwellenwerte
thresholds_compare = [0.50, t_best_f1, t_best_mcc]
print("\n=== Vergleich Schwellenwerte ===")
print(f"{'Schwellenwert':15s} | {'F1':6s} | {'MCC':6s} | "
      f"{'P1':6s} | {'R1':6s} | {'FP':4s} | {'FN':4s}")
print("-" * 65)
for t in thresholds_compare:
    preds = (oof_probs >= t).astype(int)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    f1 = f1_score(y, preds, average="macro")
    mcc = matthews_corrcoef(y, preds)
    p1 = precision_score(y, preds, zero_division=0)
    r1 = recall_score(y, preds, zero_division=0)
    print(f"{t:15.2f} | {f1:.4f} | {mcc:.4f} | "
          f"{p1:.4f} | {r1:.4f} | {fp:4d} | {fn:4d}")

# Threshold-Ergebnisse speichern
threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "f1_macro": f1_scores,
    "mcc": mcc_scores,
    "precision_1": precision_scores,
    "recall_1": recall_scores,
})
threshold_df.to_csv(
    "../reports/tables/05_threshold_optimization.csv", index=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# F1 + MCC vs. Threshold
axes[0].plot(thresholds, f1_arr, color=COLOR_GOLD,
             linewidth=2, label="F1 Macro")
axes[0].plot(thresholds, mcc_arr, color=COLOR_BLUE,
             linewidth=2, label="MCC")
axes[0].axvline(0.5, color=COLOR_TEXT_MUTED, linestyle="--",
                linewidth=1, label="Standard (0.50)")
axes[0].axvline(t_best_f1, color=COLOR_GOLD, linestyle=":",
                linewidth=1.5, label=f"Best F1 ({t_best_f1:.2f})")
axes[0].axvline(t_best_mcc, color=COLOR_BLUE, linestyle=":",
                linewidth=1.5, label=f"Best MCC ({t_best_mcc:.2f})")
axes[0].set_xlabel("Schwellenwert")
axes[0].set_ylabel("Score")
axes[0].set_title("F1 Macro & MCC vs. Schwellenwert")
axes[0].legend(fontsize=9)

# Precision/Recall vs. Threshold
axes[1].plot(thresholds, precision_scores, color=COLOR_GOLD,
             linewidth=2, label="Precision_1")
axes[1].plot(thresholds, recall_scores, color=COLOR_BLUE,
             linewidth=2, label="Recall_1")
axes[1].axvline(0.5, color=COLOR_TEXT_MUTED, linestyle="--",
                linewidth=1, label="Standard (0.50)")
axes[1].axvline(t_best_f1, color=COLOR_TEXT_MUTED, linestyle=":",
                linewidth=1.5, label=f"Best F1 ({t_best_f1:.2f})")
axes[1].set_xlabel("Schwellenwert")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision_1 & Recall_1 vs. Schwellenwert")
axes[1].legend(fontsize=9)

fig.suptitle("Threshold-Optimierung: LogReg_balanced Champion (Out-of-Fold)")
fig.tight_layout()

save_figure(fig, "../reports/figures/05_threshold_optimization.png")
plt.show()

## Erkenntnis: Threshold-Optimierung

**Ergebnis:**
| Schwellenwert | F1 | MCC | Precision_1 | Recall_1 | FP | FN |
|---|---|---|---|---|---|---|
| 0.50 (Standard) | 0.7775 | 0.5551 | 0.741 | 0.728 | 703 | 753 |
| **0.54 (Optimal)** | **0.7832** | **0.5709** | **0.784** | **0.686** | **522** | **869** |

**Metriken-Gewinn bei t=0.54:**
  F1 Macro: +0.0057 | MCC: +0.0158 | FP: -181

**Trade-off — der entscheidende Punkt:**
Der Gewinn bei F1/MCC kommt durch Verschiebung des Fehler-Profils:
- 181 weniger Fehlalarme (FP: 703 → 522, -25.7%)
- 116 mehr verpasste Disasters (FN: 753 → 869, +15.4%)

**Bewertung aus Disaster-Response-Perspektive:**
In einem realen Einsatz ist FN (verpasste Katastrophe) teurer
als FP (Fehlalarm). Threshold 0.54 verschlechtert diesen
anwendungsspezifischen Trade-off — das Modell verpasst mehr
echte Disasters, auch wenn F1 und MCC formal steigen.

Zwei valide operative Entscheidungen:
- t=0.50: ausgewogeneres Profil, weniger verpasste Disasters
  → empfohlen wenn FN-Kosten > FP-Kosten (Disaster-Response)
- t=0.54: besseres F1/MCC, weniger Fehlalarme
  → empfohlen wenn FP-Kosten > FN-Kosten (z.B. Ressourcenplanung)

**Wichtige Einschraenkung:**
Threshold-Optimierung auf Out-of-Fold-Daten hat einen leichten
Optimismus-Bias — der optimale Threshold auf echten Testdaten
koennte geringfuegig abweichen. Fuer Phase 6 (Fehleranalyse)
werden beide Schwellenwerte berichtet.

**Finales Champion-Modell:**
LogReg_balanced Config C, t=0.50 (Standard) als Hauptkandidat
fuer Disaster-Response-Kontext. t=0.54 als dokumentierte
Alternative fuer ressourcenoptimierte Szenarien.

In [ ]:
# =============================================================================
# Zelle 15 – Feature Importance: LogReg Koeffizienten + Permutation Importance
# =============================================================================

from sklearn.inspection import permutation_importance

# Champion auf gesamtem Datensatz trainieren
champion_full = LogisticRegression(
    max_iter=1000, random_state=42, class_weight="balanced")
champion_full.fit(X, y)

feature_names = np.array(vectorizer_C.get_feature_names_out())
coefficients = champion_full.coef_[0]

# Top-20 pro Richtung (Disaster vs. kein Disaster)
coef_df = pd.DataFrame({
    "word": feature_names,
    "coefficient": coefficients,
    "abs_coef": np.abs(coefficients)
}).sort_values("coefficient", ascending=False)

top20_disaster = coef_df.head(20)
top20_no_disaster = coef_df.tail(20).sort_values("coefficient")

print("Top 20 Woerter → Disaster (positive Koeffizienten):")
print(top20_disaster[["word", "coefficient"]].to_string(index=False))
print("\nTop 20 Woerter → Kein Disaster (negative Koeffizienten):")
print(top20_no_disaster[["word", "coefficient"]].to_string(index=False))

coef_df.to_csv("../reports/tables/05_logreg_coefficients.csv", index=False)

# Permutation Importance (Stichprobe fuer Geschwindigkeit)
# Korrektur: sparse Matrix zu dense konvertieren (Anforderung von sklearn)
print("\nPermutation Importance (n_repeats=10, Stichprobe 2000)...")
sample_idx = np.random.RandomState(42).choice(len(y), 2000, replace=False)
X_sample_dense = X[sample_idx].toarray()
y_sample = y[sample_idx]

perm_result = permutation_importance(
    champion_full, X_sample_dense, y_sample,
    n_repeats=10, random_state=42,
    scoring=make_scorer(f1_score, average="macro"),
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "word": feature_names,
    "perm_importance_mean": perm_result.importances_mean,
    "perm_importance_std": perm_result.importances_std,
}).sort_values("perm_importance_mean", ascending=False)

print("\nTop 20 Woerter nach Permutation Importance:")
print(perm_df.head(20)[["word", "perm_importance_mean",
                          "perm_importance_std"]].to_string(index=False))

perm_df.to_csv("../reports/tables/05_permutation_importance.csv", index=False)

# Plot: Koeffizienten + Permutation nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Koeffizienten-Plot
all_top = pd.concat([top20_disaster, top20_no_disaster])
colors_coef = [COLOR_CLASS_1 if c > 0 else COLOR_CLASS_0
               for c in all_top["coefficient"]]
axes[0].barh(all_top["word"], all_top["coefficient"],
             color=colors_coef, alpha=0.85)
axes[0].axvline(0, color=COLOR_TEXT_MUTED, linewidth=1)
axes[0].set_title("LogReg Koeffizienten\n"
                  "(Gold=Disaster, Blau=Kein Disaster)")
axes[0].set_xlabel("Koeffizient")

# Permutation Importance Plot (Top 20)
top20_perm = perm_df.head(20)
axes[1].barh(top20_perm["word"][::-1],
             top20_perm["perm_importance_mean"][::-1],
             xerr=top20_perm["perm_importance_std"][::-1],
             color=COLOR_GOLD, alpha=0.85, capsize=3,
             error_kw={"ecolor": COLOR_TEXT_MUTED})
axes[1].set_title("Permutation Importance\n"
                  "(F1-Verlust wenn Feature permutiert)")
axes[1].set_xlabel("F1-Verlust (Mittelwert ± Std, n=10)")

fig.suptitle("Feature Importance: LogReg_balanced Champion (Config C, Stemmed)")
fig.tight_layout()

save_figure(fig, "../reports/figures/05_feature_importance.png")
plt.show()

## Erkenntnis: Feature Importance

### LogReg Koeffizienten — direkt interpretierbar

**Top Disaster-Signalwoerter (positive Koeffizienten):**
hiroshima (3.63) > fire (3.36) > wildfir (3.10) > storm (3.03) >
flood (2.98) > california (2.91) > evacu (2.71) > kill (2.70)

**Top Kein-Disaster-Signalwoerter (negative Koeffizienten):**
love (-2.23) > bag (-1.95) > let (-1.86) > blew (-1.68) >
song (-1.56) > feel (-1.51) > want (-1.49) > scream (-1.45)

**Inhaltliche Validierung:**
- Disaster-Woerter: fast ausschliesslich eindeutige Katastrophen-
  begriffe (hiroshima, typhoon, derail, massacr, drought) —
  das Modell hat die relevanten semantischen Felder korrekt
  identifiziert
- Kein-Disaster-Woerter: typische Alltagssprache/Social-Media
  (love, song, feel, want, scream als Ausdruck) — das Modell
  erkennt umgangssprachliche, metaphorische Nutzung korrekt

### Permutation Importance — modell-agnostisch

| Rang | Wort | F1-Verlust | Std |
|---|---|---|---|
| 1 | fire | 0.0105 | 0.0017 |
| 2 | storm | 0.0075 | 0.0012 |
| 3 | hiroshima | 0.0059 | 0.0008 |
| 4 | num | 0.0047 | 0.0022 |
| 5 | wildfir | 0.0039 | 0.0011 |

### Kritischer Befund: Gleichmaessige Feature-Verteilung

**Vergleich mit dem Vorprojekt (Loan Status):**
- Loan Status: credit_history allein erklaert ~60% der Entscheidung
  (Permutation Importance: 0.127 PR-AUC-Verlust)
- Disaster Tweets: staerkstes Feature "fire" erklaert nur ~1.3%
  des F1 (0.0105 Verlust). Verhaeltnis Top1/Top20: nur ~7x
  (vs. ~100x beim Loan-Modell)

**Das bedeutet:** Kein einzelnes "Killer-Feature" vorhanden —
Information ist ueber viele schwache, kontextabhaengige Signale
verteilt. Das Modell benoetigt das Zusammenspiel vieler Woerter
fuer zuverlaessige Klassifikation.

**Wichtige Einschraenkung der Permutation Importance:**
Hohe relative Std (z. B. "num": ±47%) auf der 2000-Sample-
Stichprobe zeigt Rauschen in den Schaetzungen. Grobe Richtung
ist zuverlaessig, genaue Ranking-Position nicht.

**"num" als Feature (Rang 4):** Der NUM-Platzhalter aus
Notebook 02 ist ein starkes Feature — Tweets mit Zahlen
(z.B. Opferzahlen, Koordinaten, Erdbeben-Magnituden) sind
haeufiger Disaster-Tweets. Bestaetigt rueckwirkend die
Entscheidung, Zahlen durch einen Platzhalter zu ersetzen
statt zu entfernen.

### Verbindung zum SOTA-Abstand

Die gleichmaessige Feature-Verteilung ist die tiefste Ursache
des Abstands zu SOTA (F1 ~0.05-0.07 unter BERT-Klasse):
TF-IDF behandelt jedes Wort isoliert. Bei vielen schwachen,
kontextabhaengigen Signalen (wie hier) sind Transformer klar
im Vorteil — sie kombinieren schwache Signale mit globalem
Kontextverstaendnis (Self-Attention ueber den gesamten Tweet).
Disaster Tweets erfordern genau dieses Kontextverstaendnis
(Metaphern, Ironie, Alltagssprache vs. echte Katastrophe).

In [ ]:
# =============================================================================
# Zelle 17 – Champion-Modell speichern + Registry aktualisieren
# =============================================================================
# Champion: LogReg_balanced Config C (Begruendung: Zelle 12/16)
# Gespeichert wird das auf GESAMTEM Datensatz trainierte Modell
# (champion_full aus Zelle 15 — bereits gefittet)

import json

# Champion-Metadaten
champion_metadata = {
    "modell": "LogisticRegression",
    "class_weight": "balanced",
    "C": 1.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "vectorizer": "TF-IDF Config C (text_stemmed, min_df=2)",
    "threshold_standard": 0.50,
    "threshold_optimiert": 0.54,
    "cv_f1_macro": 0.7775,
    "cv_f1_std": 0.0069,
    "cv_f1_ci_low": 0.768,
    "cv_f1_ci_high": 0.788,
    "cv_roc_auc": 0.8487,
    "cv_mcc": 0.5551,
    "overfitting_gap": 0.099,
    "trainingsdaten": 6801,
    "features": 4869,
}

# Modell speichern
joblib.dump(champion_full, "../models/champion_logreg.joblib")
joblib.dump(vectorizer_C, "../models/champion_vectorizer.joblib")

# Metadaten als JSON
with open("../models/champion_metadata.json", "w") as f:
    json.dump(champion_metadata, f, indent=2)

print("Champion-Modell gespeichert:")
print("  models/champion_logreg.joblib")
print("  models/champion_vectorizer.joblib")
print("  models/champion_metadata.json")

# Registry aktualisieren
registry_champion = """
## Champion-Modell (Phase 5)

| Datei | Inhalt |
|---|---|
| champion_logreg.joblib | LogReg_balanced, auf 6801 Trainingsdaten trainiert |
| champion_vectorizer.joblib | TF-IDF Config C (text_stemmed, min_df=2, 4869 Features) |
| champion_metadata.json | Alle Parameter und CV-Metriken |

### Champion-Metriken (5-Fold Stratified CV)
- F1 Macro:  0.7775 ± 0.0069 [95% KI: 0.768 – 0.788]
- ROC-AUC:   0.8487 ± 0.0122
- MCC:       0.5551
- Gap:       0.099
- Threshold: 0.50 (Standard) / 0.54 (F1-optimiert)

### Entscheidungsbegruendung
Kein Modell ist statistisch signifikant besser (alle F1-Unterschiede
< 0.54x Std). LogReg gewaehlt wegen: hoechster ROC-AUC, ausgewogenstes
Fehler-Profil (P1=0.741/R1=0.728), beste Erklaerbarkeit, robustes
Tuning-Verhalten (Default-Parameter optimal).
"""

with open("../models/registry.md", "a") as f:
    f.write(registry_champion)
print("\nregistry.md aktualisiert")

# Kurze Verifikation: Modell neu laden und Probe-Vorhersage
champion_loaded = joblib.load("../models/champion_logreg.joblib")
vectorizer_loaded = joblib.load("../models/champion_vectorizer.joblib")

test_tweets = [
    "forest fire near La Ronge Sask Canada",
    "I am so devastated my team lost the game",
    "Earthquake magnitude 6.2 hits central Italy killing 5",
]

from preprocessing import clean_text, tokenize, remove_stopwords, apply_stemming

for tweet in test_tweets:
    cleaned = " ".join(apply_stemming(
        remove_stopwords(tokenize(clean_text(tweet)))))
    X_test_tweet = vectorizer_loaded.transform([cleaned])
    prob = champion_loaded.predict_proba(X_test_tweet)[0][1]
    pred = "DISASTER" if prob >= 0.5 else "KEIN DISASTER"
    print(f"\nTweet:      {tweet}")
    print(f"Bereinigt:  {cleaned}")
    print(f"P(Disaster)={prob:.3f} → {pred}")

# Abschluss: Notebook 05 — Hyperparameter-Tuning

## Champion-Modell

**LogReg_balanced Config C** (TF-IDF, text_stemmed, min_df=2)
  F1 Macro:  0.7775 ± 0.0069 [95% KI: 0.768 – 0.788]
  ROC-AUC:   0.8487 ± 0.0122
  MCC:       0.5551
  Gap:       0.099
  Threshold: 0.50 (Standard) | 0.54 (F1/MCC-optimiert)

## Tuning-Bilanz

| Modell | Baseline F1 | Getunt F1 | Delta | Erkennnis |
|---|---|---|---|---|
| LogReg_balanced | 0.7775 | 0.7782 | +0.0007 | Default optimal |
| MultinomialNB | 0.7799 | 0.7786 | -0.0013 | Nested CV korrigiert Bias |
| LinearSVC | 0.7681 | 0.7807 | **+0.0126** | Groesstes Tuning-Potenzial |

## Kernerkenntnisse

1. **Statistisch: kein Modell ist signifikant besser**
   Alle F1-Unterschiede < 0.54× Std, KI ueberlappen vollstaendig.
   Champion-Entscheidung basiert auf nicht-F1-Kriterien:
   ROC-AUC, Fehler-Profil, Erklaerbarkeit, Tuning-Robustheit.

2. **Grid Search bestaetigt Default-Parameter bei LogReg:**
   C=1.0, L2, lbfgs war bereits optimal — kein Selektions-Bias
   durch Tuning. Baseline-Ergebnisse aus Phase 4 zuverlaessig.

3. **LinearSVC: Bias-Variance-Theorie empirisch bestaetigt:**
   Groesster Baseline-Gap (0.181) → groesster Tuning-Gewinn
   (+0.0126 F1, Gap -0.064). Ursache des Baseline-Overfittings
   war Konvergenzproblem (max_iter=1000), nicht falsches C.

4. **Threshold-Optimierung: Trade-off, kein klarer Gewinner:**
   t=0.54 verbessert F1/MCC (+0.006/+0.016), aber erhoehte FN
   (+116 verpasste Disasters). Fuer Disaster-Response t=0.50
   bevorzugt (FN teurer als FP). Beide dokumentiert.

5. **Feature Importance: viele schwache Signale, kein Killer-Feature:**
   Staerkstes Feature "fire" erklaert nur ~1.3% des F1 (vs.
   ~60% bei credit_history im Loan-Projekt). Gleichmaessige
   Verteilung ist strukturelle Grenze von TF-IDF bei diesem
   Datensatz — direkte Begruendung fuer Bonus-Branch (Transformer).

6. **Probe-Vorhersage live demonstriert:**
   "I am so devastated my team lost the game" → P(Disaster)=0.504
   Modell am Rande des Zufalls — "devast" hat hohen Disaster-
   Koeffizienten, Kontext (Sport, Metapher) wird nicht erkannt.
   Paradigmatisches Beispiel fuer Phase 6 Fehleranalyse.

## SOTA-Abstand (aktualisiert)

| | F1 | ROC-AUC |
|---|---|---|
| Champion (LogReg) | 0.7775 | 0.8487 |
| SOTA (BERT-Klasse) | 0.83-0.85 | ~0.90 |
| Abstand | ~0.055-0.075 | ~0.05 |

Tuning hat den Abstand NICHT reduziert — er war bereits in der
Baseline strukturell bedingt. Der Abstand liegt in der Modellklasse
(TF-IDF vs. Kontextmodell), nicht in den Hyperparametern.

## Gespeicherte Artefakte

- `models/champion_logreg.joblib` — finales Modell
- `models/champion_vectorizer.joblib` — finaler Vectorizer
- `models/champion_metadata.json` — alle Parameter/Metriken